#**Homework 2 - From RNNs to Attention**

In this homework you will build a **sequence-to-sequence (seq2seq)** pipeline that converts dates from human-readable format (`DD/MM/YYYY`) to ISO format (`YYYY-MM-DD`).

Starting from scratch, you will generate the data, build the vocabulary, and progressively implement four models of increasing sophistication: a vanilla RNN, a weak LSTM with fixed gates, a full LSTM with learnable gates, and finally an LSTM augmented with dot-product attention.

By the end, you will be able to compare all four models side-by-side and see concretely what each architectural improvement contributes.

# Setup
Run the following cell first.

In [1]:
#Imports
import torch.nn as nn
import random
import numpy as np
import torch

from datetime import date, timedelta
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Task 1: Data Preparation (4 Points)

Before we can train any model we need data. In this task you will:
1. **Generate date piars** – random `DD/MM/YYYY → YYYY-MM-DD` examples.
2. **Split** them into train / validation / test sets.
3. **Build vocabularies** (`src_stoi`, `src_itos`, `tgt_stoi`, `tgt_itos`) at the **character level**.
4. **Add special tokens**: `<pad>`, `<bos>` (beginning-of-sequence), `<eos>` (end-of-sequence).
5. **Encode** sequences to integer IDs and **pad** them to a fixed length.
6. **Wrap** everything in a PyTorch `Dataset` and create `DataLoader`s.

All subsequent tasks depend on `src_stoi`, `tgt_stoi`, `tgt_itos`, `train_loader`, and `val_loader`.So make sure those names are correct.

## Task 1.1: Date Pair Generation (0.5 Points)

Implement `generate_date_pairs(n)` that returns a list of `n` tuples `(src, tgt)` where:
- `src` is a date string in `DD/MM/YYYY` format (e.g. `'24/12/2025'`)
- `tgt` is the same date in `YYYY-MM-DD` format (e.g. `'2025-12-24'`)

Make sure you only generate **valid** calendar dates (e.g. no 30 February).

```
generate_date_pairs(3)
# Example output:
# [('07/03/2011', '2011-03-07'), ('19/11/1998', '1998-11-19'), ...]
```

In [2]:
import random
from datetime import datetime, timedelta
def generate_date_pairs(n: int):
    # TODO: Generate n random (src, tgt) date string pairs
    pairs = []
    for _ in range (n):
      start_date = datetime(1950, 1, 1)
      random_days = random.randint(0,30000)
      date = start_date + timedelta(days = random_days)
      src = date.strftime("%d/%m/%Y")
      tgt = date.strftime("%Y-%m-%d")
      pairs.append((src,tgt))
    return pairs



In [3]:
#Sanity check
samples = generate_date_pairs(5)
for src, tgt in samples:
    print(src, "=>", tgt)

14/05/2007 => 2007-05-14
28/12/1959 => 1959-12-28
30/03/1952 => 1952-03-30
12/07/2016 => 2016-07-12
04/09/1974 => 1974-09-04


## Task 1.2: Train / Val / Test Split (0.5 Points)

Generate **10 000** date pairs in total and split them:


Suggested Split:
- **train_pairs**: first 8 000
- **val_pairs**: next 1 000
- **test_pairs**: last 1 000

Store them in variables named exactly `train_pairs`, `val_pairs`, and `test_pairs`.

In [4]:
all_pairs = generate_date_pairs(10_000)

#TODO: Implement the train, val, and test split
train_pairs = all_pairs[:8000]
val_pairs   = all_pairs[8000:9000]
test_pairs  = all_pairs[9000:]


In [5]:
#Sanity check
print(f"Train: {len(train_pairs)}  Val: {len(val_pairs)}  Test: {len(test_pairs)}")
#expected: Train: 8000  Val: 1000  Test: 1000

Train: 8000  Val: 1000  Test: 1000


## Task 1.3: Vocabulary Building (1 Point)

Build **character-level** vocabularies for the source side (`DD/MM/YYYY`) and the target side (`YYYY-MM-DD`).

Each vocabulary must include the three special tokens **`<pad>`**, **`<bos>`**, **`<eos>`** — and they must occupy indices **0, 1, 2** respectively.

You must produce four mappings:
- `src_stoi` – source char → index
- `src_itos` – index → source char
- `tgt_stoi` – target char → index
- `tgt_itos` – index → target char

**Hint:** The source alphabet is `{0-9, /}` and the target alphabet is `{0-9, -}`.

In [6]:
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>"]
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2

#TODO: Collect all unique characters that appear in source strnigs
src_chars = "0123456789/"
#TODO: src_stoi: special tokens first (at indices 0,1,2), then sorted chars
src_stoi = {"<pad>" : 0,
             "<bos>" : 1,
             "<eos>" : 2,
             "/" : 3,
             "0" : 4,
             "1" : 5,
             "2" : 6,
             "3" : 7,
             "4" : 8,
             "5" : 9,
             "6" : 10,
             "7" : 11,
             "8" : 12,
             "9" : 13
             }

#TODO: Build src_itos as the inverse mapping
src_itos = {idx: ch for ch, idx in src_stoi.items()}

#TODO: Collect all unique characters that appear in target strings
# Your code here
tgt_chars = "0123456789-"

#TODO: Build tgt_stoi and tgt_itos
tgt_stoi = {"<pad>" : 0,
             "<bos>" : 1,
             "<eos>" : 2,
             "-" : 3,
             "0" : 4,
             "1" : 5,
             "2" : 6,
             "3" : 7,
             "4" : 8,
             "5" : 9,
             "6" : 10,
             "7" : 11,
             "8" : 12,
             "9" : 13
             }

tgt_itos = {idx: ch for ch, idx in tgt_stoi.items()}



In [7]:
# Sanity Check
# expected:
# Source vocab: {'<pad>': 0, '<bos>': 1, '<eos>': 2, '/': 3, '0': 4, '1': 5, '2': 6, '3': 7, '4': 8, '5': 9, '6': 10, '7': 11, '8': 12, '9': 13}
# Target vocab: {'<pad>': 0, '<bos>': 1, '<eos>': 2, '-': 3, '0': 4, '1': 5, '2': 6, '3': 7, '4': 8, '5': 9, '6': 10, '7': 11, '8': 12, '9': 13}
print("Source vocab:", src_stoi)
print("Target vocab:", tgt_stoi)

Source vocab: {'<pad>': 0, '<bos>': 1, '<eos>': 2, '/': 3, '0': 4, '1': 5, '2': 6, '3': 7, '4': 8, '5': 9, '6': 10, '7': 11, '8': 12, '9': 13}
Target vocab: {'<pad>': 0, '<bos>': 1, '<eos>': 2, '-': 3, '0': 4, '1': 5, '2': 6, '3': 7, '4': 8, '5': 9, '6': 10, '7': 11, '8': 12, '9': 13}


## Task 1.4: Encoding Helpers (1 Point)

Implement two helper functions used both during training and at inference time.

**`encode_source(text, stoi)`** — turns a raw source string into a list of integer IDs with `<eos>` at the end.

Example: '24/12/2025' -> [3, 5, 10, 2, 3, ..., EOS_IDX]

**`encode_target(text, stoi)`** — turns a raw target string into a list of integer IDs **wrapped** with `<bos>` at the start and `<eos>` at the end.

Example: '2025-12-24' -> [BOS_IDX, 4, 1, 3, ...., EOS_IDX]

**`ids_to_text(ids, itos)`** — converts a list of IDs back to a string, skipping `<bos>`, `<eos>`, and `<pad>` tokens.

In [8]:
def encode_source(text: str, stoi: dict) -> list:
    #TODO:
    ids = []
    for char in text:
      ids.append(stoi[char])
    ids.append(EOS_IDX)
    return ids



def encode_target(text: str, stoi: dict) -> list:
    #TODO:
    ids = []
    ids.append(BOS_IDX)
    for char in text:
      ids.append(stoi[char])
    ids.append(EOS_IDX)
    return ids



def ids_to_text(ids: list, itos: dict) -> str:
    #TODO:

    if isinstance(ids, torch.Tensor):
        ids = ids.tolist()
    chars = []
    for i in ids:
      i = int(i)
      if i in [BOS_IDX, EOS_IDX, PAD_IDX]:
        continue
      else:
        chars.append(itos[i])

    return "".join(chars)






In [9]:
# Sanity check
# Example Output:
# Source: 28/01/1950 -> [6, 12, 3, 4, 5, 3, 5, 13, 9, 4, 2]
# Target: 1950-01-28 -> [1, 5, 13, 9, 4, 3, 4, 5, 3, 6, 12, 2]
# Decoded: 1950-01-28
src_ex, tgt_ex = train_pairs[0]
print("Source:", src_ex, "->", encode_source(src_ex, src_stoi))
print("Target:", tgt_ex, "->", encode_target(tgt_ex, tgt_stoi))
print("Decoded:", ids_to_text(encode_target(tgt_ex, tgt_stoi), tgt_itos))

Source: 21/12/1971 -> [6, 5, 3, 5, 6, 3, 5, 13, 11, 5, 2]
Target: 1971-12-21 -> [1, 5, 13, 11, 5, 3, 5, 6, 3, 6, 5, 2]
Decoded: 1971-12-21


## Task 1.5: Dataset and DataLoaders (1 Point)

Implement a PyTorch `Dataset` called `DateDataset` and create `train_loader` and `val_loader`.

Each item returned by `__getitem__` must be a tuple `(src_tensor, tgt_tensor)` where both are `torch.long` tensors **padded to a fixed length** using `PAD_IDX`.

Use a **batch size of 64** and **shuffle the training loader**. Do not shuffle the validation loader.

**Hint:** The maximum source length is 11 characters. The maximum target length (including `<bos>` and `<eos>`) is 12.

In [10]:
SRC_MAX_LEN = 11  # DD/MM/YYYY is always 10 characters + <eos>
TGT_MAX_LEN = 12  # <bos> + YYYY-MM-DD (10 chars) + <eos>


class DateDataset(Dataset):
    def __init__(self, pairs, src_stoi, tgt_stoi,src_max_len=SRC_MAX_LEN, tgt_max_len=TGT_MAX_LEN):
        self.pairs=pairs
        self.src_stoi = src_stoi
        self.tgt_stoi = tgt_stoi
        self.src_max_len = src_max_len
        self.tgt_max_len = tgt_max_len
        #TODO:


    def __len__(self):
        #TODO:
        return len(self.pairs)


    def __getitem__(self, idx):
        #TODO:
        src, tgt = self.pairs[idx]
        src_tensor = torch.tensor(encode_source(src, self.src_stoi), dtype=torch.long)
        tgt_tensor = torch.tensor(encode_target(tgt, self.tgt_stoi), dtype=torch.long)
        src_tensor = torch.nn.functional.pad(src_tensor, (0, self.src_max_len - len(src_tensor)), value=PAD_IDX)
        tgt_tensor = torch.nn.functional.pad(tgt_tensor, (0, self.tgt_max_len - len(tgt_tensor)), value=PAD_IDX)
        return src_tensor, tgt_tensor


train_dataset = DateDataset(train_pairs, src_stoi, tgt_stoi)
val_dataset   = DateDataset(val_pairs,   src_stoi, tgt_stoi)

#TODO: Create DataLoaders
# use batch_size=64, shuffle train, don't shuffle val
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [11]:
# Sanity check
src_batch, tgt_batch = next(iter(train_loader))
print("src batch shape:", src_batch.shape)   # expected: (64, 11)
print("tgt batch shape:", tgt_batch.shape)   # expected: (64, 12)

src batch shape: torch.Size([64, 11])
tgt batch shape: torch.Size([64, 12])


# Task 2: RNN baseline  (3.5 Points)
In this task, you implement a basic sequence-to-sequence model using vanilla RNNs.

The model consists of two parts:

1. **Encoder**  
   The encoder reads the full source sequence and converts it into hidden states.  
   It returns:
   - `encoder_outputs`: hidden states for all source positions, shape `[batch_size, src_len, hidden_dim]`
   - `hidden`: the final hidden state, shape `[1, batch_size, hidden_dim]`

2. **Decoder**  
   The decoder generates the target sequence one token at a time.  
   At each step, it receives:
   - the previous target token `input_tok`, shape `[batch_size]`
   - the previous hidden state `hidden`

   It returns:
   - `logits`: raw vocabulary scores for the next token, shape `[batch_size, output_dim]`
   - the updated hidden state

In [12]:
class RNNEncoder(nn.Module):
  def __init__(self, input_dim, emb_dim, hid_dim, pad_idx):
    super().__init__()
    self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
    self.rnn = nn.RNN(emb_dim, hid_dim, batch_first=True)

  def forward(self, src):
    # src: [batch_size, src_len]
    # TODO: embed the source sequence
    embedded = self.embedding(src)
    # TODO: pass embeddings through the RNN
    encoder_outputs,hidden = self.rnn(embedded)
    # TODO: return encoder outputs and final hidden state
    return encoder_outputs, hidden

class RNNDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(emb_dim, hid_dim, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input_tok, hidden):
      # input_tok: [batch_size]
      # hidden: [1, batch_size, hidden_dim]
      # TODO: embed input_tok
      # TODO: unsqueeze to length-1 sequence
      embedded = self.embedding(input_tok)
      embedded = embedded.unsqueeze(1)
      # TODO: run one RNN step
      output, hidden = self.rnn(embedded, hidden)
      # TODO: project hidden state to vocabulary logits
      logits = self.fc_out(output.squeeze(1))
      return logits, hidden

class Seq2SeqRNN(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, src, tgt, teacher_forcing_ratio=0.5):
    batch_size, tgt_len = tgt.shape
    vocab_size = self.decoder.fc_out.out_features
    outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)

    # TODO: run the encoder

    encoder_outputs, hidden = self.encoder(src)
    # TODO: initialize decoder input with <bos>
    input_tok = tgt[:, 0]
    # TODO: autoregressive decoder loop with teacher forcing
    for t in range(1, tgt_len):
            output, hidden = self.decoder(input_tok, hidden)
            outputs[:, t-1, :] = output
            # teacher forcing decision
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_tok = tgt[:, t] if teacher_force else top1
    return outputs


## Training and evaluation helpers

In this task, you implement the training and evaluation logic for the sequence-to-sequence model.

### Sequence Prediction Setup

The model predicts the target sequence **one step ahead**:

- The decoder receives `<bos>` as the first input
- It predicts tokens for positions `1 ... T-1`

### Cross-Entropy Loss

We use `nn.CrossEntropyLoss`, which expects:

- predictions of shape: `[N, vocab_size]`
- targets of shape: `[N]`

So you must **flatten both tensors**:

- predictions → `[batch_size * (tgt_len - 1), vocab_size]`
- targets → `[batch_size * (tgt_len - 1)]`

The loss should ignore padding tokens.

In [13]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
def train_one_epoch(model, loader, optimizer, clip=1.0, teacher_forcing_ratio=0.5):
    model.train()
    total_loss = 0.0

    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()

        output = model(src, tgt, teacher_forcing_ratio=teacher_forcing_ratio)

        if isinstance(output, tuple):
            output = output[0]

        vocab_size = output.shape[-1]

        output = output.reshape(-1, vocab_size)
        target = tgt[:, 1:].reshape(-1)

        loss = criterion(output, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        total_loss += loss.item()

        # DEBUG PRINTS (will now work)
        print(loss.item())
        print(output[:5])
        print(target[:20])
        print((tgt == PAD_IDX).float().mean())

    return total_loss / len(loader)

@torch.no_grad()
def evaluate_model(model, loader):
  model.eval()
  total_loss = 0.0

  for src, tgt in loader:
    src, tgt = src.to(device), tgt.to(device)

    output = model(src, tgt, teacher_forcing_ratio=0.0)

    if isinstance(output, tuple):
      output = output[0]

    vocab_size = output.shape[-1]

    output = output.reshape(-1, vocab_size)
    target = tgt[:, 1:].reshape(-1)

    loss = criterion(output, target)

    total_loss += loss.item()

  return total_loss / len(loader)

In [14]:
#Sanity check
RNN_EMB_DIM = 32
RNN_HID_DIM = 64

rnn_model = Seq2SeqRNN(
    RNNEncoder(len(src_stoi), RNN_EMB_DIM, RNN_HID_DIM, src_stoi["<pad>"]),
    RNNDecoder(len(tgt_stoi), RNN_EMB_DIM, RNN_HID_DIM, tgt_stoi["<pad>"]),
).to(device)

optimizer = torch.optim.Adam(rnn_model.parameters(), lr=1e-3)

for epoch in range(5):
  train_loss = train_one_epoch(rnn_model, train_loader, optimizer)
  val_loss = evaluate_model(rnn_model, val_loader)
  print(f"[RNN] Epoch {epoch+1}: train={train_loss:.4f} val={val_loss:.4f}")

Streaming output truncated to the last 5000 lines.
          1.1048,  1.0221,  1.2782,  1.6326,  0.9188, -0.2504],
        [-1.9307, -1.5100, -0.6728,  6.7192,  2.7706,  0.9715, -0.0264, -0.9999,
         -0.6794, -3.1221, -2.8684, -3.3838, -3.1844, -0.1693]],
       device='cuda:0', grad_fn=<SliceBackward0>)
tensor([ 5, 13, 13,  7,  3,  4, 11,  3,  6,  4,  2,  5, 13, 10, 10,  3,  4,  5,
         3,  5], device='cuda:0')
tensor(0., device='cuda:0')
0.7876507043838501
tensor([[-1.5643e+00, -1.6975e+00, -1.0123e-01,  1.6008e+00,  1.3300e-01,
          1.2194e+00,  6.5270e+00,  1.5769e-01, -6.9059e-01, -7.5639e-01,
         -9.4147e-01, -7.6298e-01, -1.6006e+00, -3.9875e-01],
        [-2.0319e+00, -1.1500e+00, -1.9102e+00,  1.1174e+00,  6.5319e+00,
          1.6630e+00, -1.4036e+00, -3.0156e-02, -1.5075e+00, -1.6801e+00,
         -1.5995e+00, -1.9887e+00, -2.8286e+00,  1.1702e+00],
        [-1.7612e+00, -1.6222e+00, -3.0160e+00, -1.6282e+00,  3.4712e+00,
          4.6990e+00,  3.0384e+00,

# Task 3: Hard-gated LSTM (2.5 Points)
In this task, you implement a simplified LSTM-like encoder-decoder model.

The goal is to introduce the idea of an LSTM memory cell.

Unlike the vanilla RNN, this model keeps two states:

- `hidden_state`: the short-term state used for prediction
- `cell_state`: the long-term memory state

We use three gates:

- `f_t = 0.5`: forget gate

- `i_t = 0.5`: input gate  

- `o_t = 1.0`: output gate

In [15]:
class WeakLSTMEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
        self.x_proj = nn.Linear(emb_dim, hid_dim)
        self.h_proj = nn.Linear(hid_dim, hid_dim)

    def forward(self, src):
        emb = self.embedding(src)
        batch_size, src_len, _ = emb.shape

        hidden_state = torch.zeros(batch_size, self.h_proj.out_features, device=src.device)
        cell_state = torch.zeros_like(hidden_state)
        outputs = []

        f_t = 0.5
        i_t = 0.5
        o_t = 1.0

        for t in range(src_len):
            x_t = emb[:, t, :]
            # TODO: compute candidate state g_t
            g_t = self.x_proj(x_t) + self.h_proj(hidden_state)
            # TODO: update c using fixed gates
            c_t = f_t * cell_state + i_t * g_t
            # TODO: update h using c and o_t
            h_t = o_t * torch.tanh(c_t)
            # TODO: store h in outputs
            outputs.append(h_t)

        outputs = torch.stack(outputs, dim=1)
        return outputs, (hidden_state.unsqueeze(0), cell_state.unsqueeze(0))

class WeakLSTMDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.x_proj = nn.Linear(emb_dim, hid_dim)
        self.h_proj = nn.Linear(hid_dim, hid_dim)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input_tok, hidden, cell):
        emb = self.embedding(input_tok)

        f_t = 0.5
        i_t = 0.5
        o_t = 1.0

        h_prev = hidden.squeeze(0)
        c_prev = cell.squeeze(0)

        # TODO: compute candidate state
        g_t = self.x_proj(emb) + self.h_proj(h_prev)
        # TODO: update c_t
        c_t = f_t * c_prev + i_t * g_t
        # TODO: update h_t
        h_t = o_t * torch.tanh(c_t)
        # TODO: project h_t to logits and restore [1,B,H] shape
        logits = self.fc_out(h_t)
        h_t = h_t.unsqueeze(0)
        c_t = c_t.unsqueeze(0)
        return logits, h_t, c_t


class Seq2SeqWeakLSTM(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)

        encoder_outputs, (hidden, cell) = self.encoder(src)
        input_tok = tgt[:, 0]

        for t in range(1, tgt_len):
          # TODO: run the weak LSTM decoder for one step
          logits, hidden, cell = self.decoder(input_tok, hidden, cell)
          # TODO: store logits
          outputs[:, t-1, :] = logits
          # TODO: teacher forcing update
          teacher_force = random.random() < teacher_forcing_ratio
          top1 = logits.argmax(1)
          input_tok = tgt[:, t] if teacher_force else top1


        return outputs

In [16]:
#Sanity Check
WEAK_EMB_DIM = 32
WEAK_HID_DIM = 32

weak_lstm_model = Seq2SeqWeakLSTM(
    WeakLSTMEncoder(len(src_stoi), WEAK_EMB_DIM, WEAK_HID_DIM, src_stoi["<pad>"]),
    WeakLSTMDecoder(len(tgt_stoi), WEAK_EMB_DIM, WEAK_HID_DIM, tgt_stoi["<pad>"]),
).to(device)

optimizer = torch.optim.Adam(weak_lstm_model.parameters(), lr=1e-3)

for epoch in range(5):
    train_loss = train_one_epoch(weak_lstm_model, train_loader, optimizer)
    val_loss = evaluate_model(weak_lstm_model, val_loader)
    print(f"[Weak LSTM] Epoch {epoch+1}: train={train_loss:.4f} val={val_loss:.4f}")

Streaming output truncated to the last 5000 lines.
         3,  6], device='cuda:0')
tensor(0., device='cuda:0')
1.479394793510437
tensor([[-1.0726, -0.8339, -0.9573, -1.6852,  1.4139,  4.0740,  3.5663, -0.8941,
         -1.7595, -1.0085, -1.4103, -1.1560, -1.1660,  0.3701],
        [-0.8917, -0.9643, -1.4060, -1.1898,  2.8582,  1.0279,  1.0796, -1.0061,
         -1.2870, -0.0774, -0.2399, -0.0425, -0.2208,  3.9354],
        [-1.3967, -1.6267, -2.9443, -0.7965,  1.1512,  0.7139,  0.8903,  0.0324,
          0.0668,  0.9214,  1.2675,  1.3351,  1.4184,  1.7092],
        [-1.9547, -2.1917, -3.0390,  1.2383,  0.7292,  0.8356,  0.3195,  0.3079,
          0.2665,  0.9626,  1.0963,  1.2213,  1.0471,  0.4845],
        [-2.4847, -2.3677, -2.4809,  3.8272,  1.3454,  1.0139, -0.0225, -0.1746,
         -0.3116,  0.2968, -0.0789, -0.0045, -0.1691, -0.2745]],
       device='cuda:0', grad_fn=<SliceBackward0>)
tensor([6, 4, 4, 9, 3, 4, 5, 3, 4, 5, 2, 6, 4, 5, 6, 3, 5, 6, 3, 5],
       device='cuda:0')


# Task 4: Soft-gated LSTM (3 Points)
In the previous task, the LSTM-like model used fixed gates such as `0.5` and `1.0`.  

In this task, you implement a stronger LSTM where the gates are **learnable**.

At each time step, the model uses:

- the current input embedding
- the previous hidden state

These are concatenated into:

`gate_input = [input_t ; hidden_state]`

The model then computes four values:

- `f_t`: forget gate  
- `i_t`: input gate  
- `o_t`: output gate  
- `g_t`: candidate memory content  

In [17]:
class LearnableLSTMEncoder(nn.Module):
  def __init__(self, input_dim, emb_dim, hid_dim, pad_idx):
    super().__init__()
    self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
    self.f_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
    self.i_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
    self.o_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
    self.g_gate = nn.Linear(emb_dim + hid_dim, hid_dim)

  def forward(self, src):
    emb = self.embedding(src)
    batch_size, src_len, _ = emb.shape
    hidden_dim = self.f_gate.out_features

    hidden_state = torch.zeros(batch_size, hidden_dim, device=src.device)
    cell_state = torch.zeros(batch_size, hidden_dim, device=src.device)
    outputs = []

    for t in range(src_len):
      input_t = emb[:, t, :]
      gate_input = torch.cat([input_t, hidden_state], dim=-1)
      f_t = torch.sigmoid(self.f_gate(gate_input))
      i_t = torch.sigmoid(self.i_gate(gate_input))
      o_t = torch.sigmoid(self.o_gate(gate_input))
      g_t = torch.tanh(self.g_gate(gate_input))

      cell_state = f_t * cell_state + i_t * g_t
      hidden_state = o_t * torch.tanh(cell_state)
      outputs.append(hidden_state)

    outputs = torch.stack(outputs, dim=1)
    return outputs, (hidden_state.unsqueeze(0), cell_state.unsqueeze(0))

class LearnableLSTMDecoder(nn.Module):
  def __init__(self, output_dim, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.f_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
        self.i_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
        self.o_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
        self.g_gate = nn.Linear(emb_dim + hid_dim, hid_dim)
        self.fc_out = nn.Linear(hid_dim, output_dim)

  def forward(self, input_tok, hidden, cell):
    emb = self.embedding(input_tok)
    h_prev = hidden.squeeze(0)
    c_prev = cell.squeeze(0)
    gate_input = torch.cat([emb, h_prev], dim=-1)

    f_t = torch.sigmoid(self.f_gate(gate_input))
    i_t = torch.sigmoid(self.i_gate(gate_input))
    o_t = torch.sigmoid(self.o_gate(gate_input))
    g_t = torch.tanh(self.g_gate(gate_input))

    c_t = f_t * c_prev + i_t * g_t
    h_t = o_t * torch.tanh(c_t)
    logits = self.fc_out(h_t)
    return logits, h_t.unsqueeze(0), c_t.unsqueeze(0)


class Seq2SeqLearnableLSTM(nn.Module):
  def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

  def forward(self, src, tgt, teacher_forcing_ratio=0.5):
    batch_size, tgt_len = tgt.shape
    vocab_size = self.decoder.fc_out.out_features
    outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)

    encoder_outputs, (hidden, cell) = self.encoder(src)
    input_token = tgt[:, 0]

    for t in range(1, tgt_len):
      logits, hidden, cell = self.decoder(input_token, hidden, cell)
      outputs[:, t - 1] = logits

      teacher_force = random.random() < teacher_forcing_ratio
      top1 = logits.argmax(dim=1)
      input_token = tgt[:, t] if teacher_force else top1

    return outputs

In [18]:
#Sanity check
LSTM_EMB_DIM = 32
LSTM_HID_DIM = 64

learnable_lstm_model = Seq2SeqLearnableLSTM(
    LearnableLSTMEncoder(len(src_stoi), LSTM_EMB_DIM, LSTM_HID_DIM, src_stoi["<pad>"]),
    LearnableLSTMDecoder(len(tgt_stoi), LSTM_EMB_DIM, LSTM_HID_DIM, tgt_stoi["<pad>"]),
).to(device)

optimizer = torch.optim.Adam(learnable_lstm_model.parameters(), lr=1e-3)

for epoch in range(5):
    train_loss = train_one_epoch(learnable_lstm_model, train_loader, optimizer)
    val_loss = evaluate_model(learnable_lstm_model, val_loader)
    print(f"[Learnable LSTM] Epoch {epoch+1}: train={train_loss:.4f} val={val_loss:.4f}")

Streaming output truncated to the last 5000 lines.
        [-1.9388, -2.6267, -3.8445, -2.4031, -0.1070,  2.8918, -0.7252,  0.0297,
          1.2787,  3.6744,  2.2185,  2.5039,  3.0552,  8.2928],
        [-2.0063, -2.2873, -3.0152, -1.5695, -0.9189,  1.5258, -0.8873,  0.1948,
          1.3515,  5.2362,  3.2084,  3.3054,  3.9288,  4.5864],
        [-2.9533, -2.7601, -2.6093,  0.6277, -0.8705, -0.0738,  0.0182,  0.5417,
          1.5459,  4.4837,  2.4266,  2.5515,  2.3710,  2.1859],
        [-3.6614, -3.1625, -2.5035,  5.4826,  0.9233, -0.0527, -0.0262,  0.6811,
          0.7447,  0.3045, -0.6042, -0.2597, -0.8724, -1.0715]],
       device='cuda:0', grad_fn=<SliceBackward0>)
tensor([ 5, 13, 13,  9,  3,  4,  8,  3,  6, 11,  2,  6,  4,  7,  4,  3,  5,  6,
         3,  5], device='cuda:0')
tensor(0., device='cuda:0')
0.6662254333496094
tensor([[-3.0220, -2.8036, -4.0408, -3.1740,  1.7724,  7.7290,  3.3894,  0.9640,
         -0.4538, -0.5640, -0.3191, -0.1913, -0.3331,  2.8022],
        [-2.

# Task 5: Add attention   (3 Points)
So far, the decoder only received the final hidden and cell states from the encoder.

This creates a bottleneck

In this task, you add an attention mechanism.

### Dot-Product Attention

The attention module receives:

- `query`
- `encoder_outputs`

It should compute:

1. **Attention scores**  

2. **Attention weights**  

3. **Context vector**

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

class DotAttention(nn.Module):
    def forward(self, query, encoder_outputs):
        # query: [batch_size, hidden_dim]
        # encoder_outputs: [batch_size, src_len, hidden_dim]

        # TODO: compute dot-product attention score
        scores = torch.bmm(encoder_outputs, query.unsqueeze(2)).squeeze(2)
        # TODO: normalize scores with softmax
        attn_weights = F.softmax(scores, dim=-1)
        # TODO: compute context vector as weighted sum of encoder outputs
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)

        return context, attn_weights

class AttnLearnableLSTMDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.attention = DotAttention()
        self.f_gate = nn.Linear(emb_dim + hid_dim + hid_dim, hid_dim)
        self.i_gate = nn.Linear(emb_dim + hid_dim + hid_dim, hid_dim)
        self.o_gate = nn.Linear(emb_dim + hid_dim + hid_dim, hid_dim)
        self.g_gate = nn.Linear(emb_dim + hid_dim + hid_dim, hid_dim)
        self.fc_out = nn.Linear(hid_dim + hid_dim, output_dim)

    def forward(self, input_tok, hidden, cell, encoder_outputs):
        emb = self.embedding(input_tok)
        h_prev = hidden.squeeze(0)
        c_prev = cell.squeeze(0)

        # TODO: get context and attention weights
        context, attn_weights = self.attention(h_prev, encoder_outputs)
        # TODO: concatenate embedding, previous hidden state, and context
        lstm_input = torch.cat((emb, h_prev, context), dim=1)
        # TODO: compute learnable gates
        f_t = torch.sigmoid(self.f_gate(lstm_input))
        i_t = torch.sigmoid(self.i_gate(lstm_input))
        o_t = torch.sigmoid(self.o_gate(lstm_input))
        g_t = torch.tanh(self.g_gate(lstm_input))
        # TODO: update c_t and h_t
        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        # TODO: project [h_t ; context] to logits
        prediction = self.fc_out(torch.cat((h_t, context), dim=1))

        return prediction, h_t.unsqueeze(0), c_t.unsqueeze(0), attn_weights

class Seq2SeqLSTMAttn(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)
        all_attn = []

        encoder_outputs, (hidden, cell) = self.encoder(src)
        input_tok = tgt[:, 0]

        for t in range(1, tgt_len):


            # TODO: decoder step with attention
            output, hidden, cell, attn_weights = self.decoder(input_tok, hidden, cell, encoder_outputs)
            # TODO: store logits and attention weights
            outputs[:, t - 1, :] = output
            all_attn.append(attn_weights)
            # TODO: teacher forcing update
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_tok = tgt[:, t] if teacher_force else top1

        all_attn = torch.stack(all_attn, dim=1)
        return outputs, all_attn

In [20]:
#Sanity check
attn_model = Seq2SeqLSTMAttn(
    LearnableLSTMEncoder(len(src_stoi), LSTM_EMB_DIM, LSTM_HID_DIM, src_stoi["<pad>"]),
    AttnLearnableLSTMDecoder(len(tgt_stoi), LSTM_EMB_DIM, LSTM_HID_DIM, tgt_stoi["<pad>"]),
).to(device)

optimizer = torch.optim.Adam(attn_model.parameters(), lr=1e-3)

for epoch in range(5):
    train_loss = train_one_epoch(attn_model, train_loader, optimizer)
    val_loss = evaluate_model(attn_model, val_loader)
    print(f"[LSTM + Attention] Epoch {epoch+1}: train={train_loss:.4f} val={val_loss:.4f}")

Streaming output truncated to the last 5000 lines.
         3,  7], device='cuda:0')
tensor(0., device='cuda:0')
0.05434659868478775
tensor([[-3.9317, -3.4710, -6.3848, -0.6428,  3.6772,  7.5595,  9.3830,  0.1889,
         -4.7576, -3.0385, -2.4743, -3.3297, -2.7771, -0.3971],
        [-3.8655, -3.6273, -6.9089, -1.6501,  7.9180,  5.9885,  2.6000,  1.9592,
         -2.5297, -1.1356, -0.5955, -2.8836, -1.8182,  0.2147],
        [-4.3147, -4.4494, -5.8569,  0.3197,  3.5533,  8.8768,  3.0501,  2.0840,
         -2.2243, -1.5228, -1.0124, -1.9883, -2.2096, -0.0815],
        [-4.7510, -4.9724, -5.3091,  2.8554,  2.5838,  6.3951,  2.0137,  1.9521,
         -1.7346, -1.1967, -0.1988, -1.2297, -1.1931,  0.2022],
        [-4.8828, -5.2637, -2.4954,  8.1880,  2.1352,  4.2669,  2.2002,  0.4994,
         -0.9050, -2.7241, -1.4842, -1.6717, -2.1884, -2.4660]],
       device='cuda:0', grad_fn=<SliceBackward0>)
tensor([ 6,  4,  5,  5,  3,  4, 12,  3,  5,  8,  2,  6,  4,  6,  9,  3,  5,  6,
         3,

# Task 6: Inference  (1 Point)
At inference time we don't have the ground-truth target sequence, so we generate one token at a time.


In [21]:
@torch.no_grad()
def greedy_decode(model, src_text, src_stoi, tgt_stoi, tgt_itos, max_len=16, attention=False):
    model.eval()

    src_ids = encode_source(src_text, src_stoi)
    src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)

    generated = [tgt_stoi["<bos>"]]
    attn_map = []

    if attention:
        encoder_outputs, (hidden, cell) = model.encoder(src)
    else:
        encoder_outputs, hidden_or_state = model.encoder(src)

    input_tok = torch.tensor([tgt_stoi["<bos>"]], device=device)

    for _ in range(max_len):
        if attention:
            output, hidden, cell, attn_weights = model.decoder(input_tok, hidden, cell, encoder_outputs)
            attn_map.append(attn_weights.squeeze(0).squeeze(0).cpu().tolist())
        else:
            if isinstance(model, Seq2SeqRNN):
                output, hidden_or_state = model.decoder(input_tok, hidden_or_state)
            else:
                hidden, cell = hidden_or_state
                output, hidden, cell = model.decoder(input_tok, hidden, cell)
                hidden_or_state = (hidden, cell)

        next_tok = output.argmax(1).item()
        generated.append(next_tok)

        if next_tok == tgt_stoi["<eos>"]:
            break

        input_tok = torch.tensor([next_tok], device=device)

    decoded = ids_to_text(generated, tgt_itos)
    return decoded, attn_map

## Sanity check: Model Comparison
Use the following cell to compare predictions from all four models.

In [22]:
examples = [
    "01/01/2001",
    "24/12/2025",
    "13/05/1999",
    "07/11/2018",
]

for src_text in examples:
    pred_rnn, _ = greedy_decode(rnn_model, src_text, src_stoi, tgt_stoi, tgt_itos, attention=False)
    pred_weak, _ = greedy_decode(weak_lstm_model, src_text, src_stoi, tgt_stoi, tgt_itos, attention=False)
    pred_lstm, _ = greedy_decode(learnable_lstm_model, src_text, src_stoi, tgt_stoi, tgt_itos, attention=False)
    pred_attn, attn = greedy_decode(attn_model, src_text, src_stoi, tgt_stoi, tgt_itos, attention=True)

    print(f"Source:      {src_text}")
    print(f"Target:      {src_text[6:10]}-{src_text[3:5]}-{src_text[0:2]}")
    print(f"RNN:         {pred_rnn}")
    print(f"Weak LSTM:   {pred_weak}")
    print(f"Learn. LSTM: {pred_lstm}")
    print(f"LSTM+Attn:   {pred_attn}")
    print("-" * 50)

Source:      01/01/2001
Target:      2001-01-01
RNN:         2001-11-27
Weak LSTM:   1998-01-19
Learn. LSTM: 2010-01-10
LSTM+Attn:   2001-01-01
--------------------------------------------------
Source:      24/12/2025
Target:      2025-12-24
RNN:         2028-11-29
Weak LSTM:   1998-01-19
Learn. LSTM: 2025-12-24
LSTM+Attn:   2025-12-24
--------------------------------------------------
Source:      13/05/1999
Target:      1999-05-13
RNN:         1999-05-11
Weak LSTM:   1998-01-19
Learn. LSTM: 1999-05-13
LSTM+Attn:   1999-05-13
--------------------------------------------------
Source:      07/11/2018
Target:      2018-11-07
RNN:         2015-10-22
Weak LSTM:   1998-01-19
Learn. LSTM: 2018-11-07
LSTM+Attn:   2018-11-07
--------------------------------------------------


# Task 7: Multiple Choice Questions (3 Points)

For each question, print your answer(s) in the code cell below it.

**Example:**
```python
print("A")
```




### Q1 (0.5 pts)

We append `<eos>` to the **source** sequence and wrap the **target** with `<bos>` ... `<eos>`. What is the main purpose of the `<eos>` token on the source side?

A. It tells the encoder to stop updating its hidden state after the last real character.  
B. It replaces the `<pad>` token so that only one special token is needed.  
C. It pads the source sequence to a fixed length.  
D. It acts as an explicit end-of-input signal that the encoder can learn to recognise, after which the final hidden state is a clean summary of the whole sequence.  

In [23]:
# Write your answer below
print("D")

D



### Q2 (0.5 pts)

We pass `ignore_index=PAD_IDX` to `nn.CrossEntropyLoss`. What is the direct effect of this setting?

A. Padding tokens are replaced with zeros before the loss is computed.  
B. The loss contribution from padding positions is set to zero, so they do not affect the gradients.  
C. The model learns to predict `<pad>` with higher confidence.  
D. It reduces the effective vocabulary size during training.



In [24]:
# Write your answer below
print("B")

B


### Q3 (0.5 pt)

What is the main problem that the LSTM was designed to solve compared to a vanilla RNN?

A. RNNs suffer from vanishing gradients, making it hard to learn long-range dependencies; the LSTM cell state provides a path for gradients to flow across many steps with minimal attenuation.  
B. RNNs cannot handle variable-length sequences, while LSTMs can.  
C. RNNs require more memory than LSTMs for the same hidden dimension.  
D. RNNs can only process sequences in one direction, while LSTMs are bidirectional by default.






In [25]:
# Write your answer below
print("A")

A


### Q4 (0.5 pts)

In the Weak LSTM the forget gate is fixed at `f = 0.5`. What does this mean for information in the cell state?

A. Exactly half of the cell-state memory is discarded at every time step, regardless of the input.  
B. The model learns to forget the 50 % of information that is least useful.  
C. The cell state is reset to zero every two time steps.  
D. The hidden state and cell state become identical after many steps.


In [26]:
# Write your answer below
print("A")

A


### Q5 (0.5 pts)

During training, teacher forcing feeds the ground-truth target token as the next decoder input instead of the model's own prediction. Which of the following is a known **disadvantage** of teacher forcing?

A. It makes training slower because the decoder must run twice per step.  
B. It can create a train-inference mismatch: at inference time the model must use its own (potentially wrong) predictions, which it was never trained to recover from.  
C. It prevents the model from learning to generate `<eos>` correctly.  
D. It increases the loss because gold tokens are harder to predict than sampled tokens.



In [27]:
# Write your answer below
print("B")

B


### Q6 (0.5 pts)


In dot-product attention, how is the context vector produced at each decoder step?

A. It is the final encoder hidden state, passed unchanged to the decoder.  
B. It is the average of all encoder outputs, weighted equally.        
C. It is a weighted sum of all encoder outputs, where the weights come from softmax-normalising the dot products between the decoder's hidden state and each encoder output.  
D. It is computed by a separate feedforward network trained independently from the seq2seq model.

In [28]:
# Write your answer below
print("C")

C
